# MSSV1_MSSV2 - PHẦN I: THU THẬP, LÀM SẠCH VÀ TIỀN XỬ LÝ DỮ LIỆU

Dataset: **Students Performance in Exams** (Kaggle) - học sinh với 5 cột phân loại và 3 cột điểm (math, reading, writing).

Chạy file này **đầu tiên**: cuối file lưu `students_cleaned.csv` cho Phần II.

In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

# ---------- settings ----------
# use the shared folder if it exists, otherwise the current folder
FILE_PATH = "../shared/StudentsPerformance.csv" if os.path.exists("../shared/StudentsPerformance.csv") else "StudentsPerformance.csv"
CHART_FOLDER = "charts"            # folder for the charts of Part III
os.makedirs(CHART_FOLDER, exist_ok=True)

TARGET = "math score"                      # the score we want to explain
GROUP_COL = "race/ethnicity"               # the main group column
NAME_COL = "parental level of education"   # no student id in this file, so regex is used on this text column

## 1.1 Thu thập và tải dữ liệu

#### Đọc dữ liệu bằng Pandas vào DataFrame.

In [2]:
df = pd.read_csv(FILE_PATH)
print("Loaded:", FILE_PATH)

Loaded: ../shared/StudentsPerformance.csv


## 1.2 Khám phá và cấu trúc cơ bản

#### Hiển thị 5 dòng đầu, 5 dòng cuối, kích thước, kiểu dữ liệu và tổng quan.

In [3]:
print(df.head())                 # first 5 rows
print(df.tail())                 # last 5 rows
print("\nShape (rows, columns):", df.shape)
print("\nData types:")
print(df.dtypes)
df.info()                        # short overview

# split the columns by type, so the names are not typed by hand
number_cols = df.select_dtypes(include="number").columns.tolist()
text_cols = df.select_dtypes(exclude="number").columns.tolist()
print("\nNumber columns:", number_cols)
print("Text columns:", text_cols)

   gender race/ethnicity parental level of education         lunch  \
0  female        group B           bachelor's degree      standard   
1  female        group C                some college      standard   
2  female        group B             master's degree      standard   
3    male        group A          associate's degree  free/reduced   
4    male        group C                some college      standard   

  test preparation course  math score  reading score  writing score  
0                    none          72             72             74  
1               completed          69             90             88  
2                    none          90             95             93  
3                    none          47             57             44  
4                    none          76             78             75  
     gender race/ethnicity parental level of education         lunch  \
995  female        group E             master's degree      standard   
996    male    

## 1.3 Kiểm tra và làm sạch dữ liệu thật

#### Kiểm tra dữ liệu thiếu và dòng trùng, rồi làm sạch.

In [4]:
raw_shape = df.shape
n_missing_real = int(df.isnull().sum().sum())     # missing values in the real data
n_duplicates = int(df.duplicated().sum())         # duplicated rows in the real data
print("Missing values per column:")
print(df.isnull().sum())
print("Duplicated rows:", n_duplicates)

# clean: keep the first copy of each duplicated row, drop rows with missing values (if any)
df = df.drop_duplicates().dropna().reset_index(drop=True)
print("Shape after cleaning:", df.shape)

display(Markdown(
    f"**Nhận xét:** Dữ liệu thật gồm {raw_shape[0]} dòng, {raw_shape[1]} cột "
    f"({len(text_cols)} cột phân loại, {len(number_cols)} cột điểm), có {n_missing_real} giá trị thiếu "
    f"và {n_duplicates} dòng trùng. Sau khi làm sạch còn {len(df)} dòng ({raw_shape[0] - len(df)} dòng bị loại)."
))

Missing values per column:
gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64
Duplicated rows: 0
Shape after cleaning: (1000, 8)


**Nhận xét:** Dữ liệu thật gồm 1000 dòng, 8 cột (5 cột phân loại, 3 cột điểm), có 0 giá trị thiếu và 0 dòng trùng. Sau khi làm sạch còn 1000 dòng (0 dòng bị loại).

## 1.4 Giả lập 5% dữ liệu thiếu và so sánh hai cách xử lý

#### Giả lập 5% thiếu ngẫu nhiên ở một cột điểm, rồi so sánh `fillna` (trung bình) với `dropna`.

In [5]:
# work on a copy, so the real data stays safe
df_dirty = df.copy()

# the column to break: the last number column that is not the target
miss_col = [c for c in number_cols if c != TARGET][-1]

# blank 5% of the rows of that column at random
miss_index = df_dirty.sample(frac=0.05, random_state=42).index
df_dirty.loc[miss_index, miss_col] = np.nan
print("Simulated 5% missing in:", miss_col)
print("\nMissing values per column:")
print(df_dirty.isnull().sum())

# method 1: fill with the mean of the column | method 2: drop the rows with missing values
df_filled = df_dirty.fillna({miss_col: df_dirty[miss_col].mean()})
df_dropped = df_dirty.dropna()
lost_pct = (1 - len(df_dropped) / len(df_dirty)) * 100
print("\nOriginal shape:", df.shape)
print("After fillna (mean):", df_filled.shape)
print("After dropna:", df_dropped.shape, "-> lost", round(lost_pct, 1), "% of the rows")

std_before, std_after = df[miss_col].std(), df_filled[miss_col].std()
std_word = "giảm" if std_after < std_before else ("tăng" if std_after > std_before else "không đổi")
display(Markdown(
    f"**Nhận xét:** Đã giả lập {int(df_dirty[miss_col].isnull().sum())} giá trị thiếu "
    f"({df_dirty[miss_col].isnull().mean() * 100:.1f}%) ở `{miss_col}`. "
    f"`fillna` (trung bình) giữ {len(df_filled)} dòng, còn `dropna` còn {len(df_dropped)} dòng, mất {lost_pct:.1f}% số dòng. "
    f"Độ lệch chuẩn của `{miss_col}` {std_word} từ {std_before:.2f} xuống {std_after:.2f} sau khi `fillna`"
    + (", vì các giá trị điền bằng trung bình không làm tăng độ phân tán. " if std_word == "giảm" else ". ")
    + "Các phần sau dùng dữ liệu thật đã làm sạch `df`, không dùng bản sao bị phá."
))

Simulated 5% missing in: writing score

Missing values per column:
gender                          0
race/ethnicity                  0
parental level of education     0
lunch                           0
test preparation course         0
math score                      0
reading score                   0
writing score                  50
dtype: int64

Original shape: (1000, 8)
After fillna (mean): (1000, 8)
After dropna: (950, 8) -> lost 5.0 % of the rows


**Nhận xét:** Đã giả lập 50 giá trị thiếu (5.0%) ở `writing score`. `fillna` (trung bình) giữ 1000 dòng, còn `dropna` còn 950 dòng, mất 5.0% số dòng. Độ lệch chuẩn của `writing score` giảm từ 15.20 xuống 14.72 sau khi `fillna`, vì các giá trị điền bằng trung bình không làm tăng độ phân tán. Các phần sau dùng dữ liệu thật đã làm sạch `df`, không dùng bản sao bị phá.

## 1.5 Cấu trúc Python cơ bản và Regex

In [ ]:
# ---------- Series -> List -> filter with basic Python ----------
score_series = df[TARGET]                  # a pandas Series
score_list = score_series.tolist()         # the same values as a Python list

# keep only the values bigger than the average, using a plain loop
average = sum(score_list) / len(score_list)
above_average = []
for value in score_list:
    if value > average:
        above_average.append(value)
print("Values in the list:", len(score_list), "| average:", round(average, 3))
print("Values above the average:", len(above_average))

Values in the list: 1000 | average: 66.089
Values above the average: 493


In [7]:
# ---------- regular expressions (re) ----------
# 1) clean the column names: keep letters, digits and underscore only
def clean_name(name):
    return re.sub(r"\W+", "_", str(name).strip()).strip("_").lower()

print("Old names:", df.columns.tolist())
df = df.rename(columns={c: clean_name(c) for c in df.columns})
print("New names:", df.columns.tolist())

# the main columns and the column lists must follow the new names
TARGET, GROUP_COL, NAME_COL = clean_name(TARGET), clean_name(GROUP_COL), clean_name(NAME_COL)
number_cols = df.select_dtypes(include="number").columns.tolist()
text_cols = df.select_dtypes(exclude="number").columns.tolist()

# 2) check the format of the NAME_COL values: only the allowed education levels
pattern = r"(some )?(high school|college)|(associate's|bachelor's|master's) degree"
is_valid = df[NAME_COL].apply(lambda s: re.fullmatch(pattern, s) is not None)
print("\nValid values in", NAME_COL + ":", is_valid.sum(), "| invalid:", (~is_valid).sum())

# 3) extract information: the degree type, for example "bachelor's degree" -> "bachelor"
degree_type = df[NAME_COL].str.extract(r"(\w+)'s degree")[0]
print("\nValues that contain a degree:", degree_type.notna().sum())
print(degree_type.value_counts())

Old names: ['gender', 'race/ethnicity', 'parental level of education', 'lunch', 'test preparation course', 'math score', 'reading score', 'writing score']
New names: ['gender', 'race_ethnicity', 'parental_level_of_education', 'lunch', 'test_preparation_course', 'math_score', 'reading_score', 'writing_score']

Valid values in parental_level_of_education: 1000 | invalid: 0

Values that contain a degree: 399
0
associate    222
bachelor     118
master        59
Name: count, dtype: int64


In [8]:
# ---------- save the cleaned real data for Part II ----------
out_path = "../shared/students_cleaned.csv" if os.path.exists("../shared/") else "students_cleaned.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path, "| shape:", df.shape)
print("Columns:", df.columns.tolist())

Saved: ../shared/students_cleaned.csv | shape: (1000, 8)
Columns: ['gender', 'race_ethnicity', 'parental_level_of_education', 'lunch', 'test_preparation_course', 'math_score', 'reading_score', 'writing_score']
